In [2]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Tuple, Dict

import numpy as np
import xarray as xr

In [3]:
@dataclass(frozen=True)
class RmmObsEofConfig:
    # Input files
    noaa_olr_file: str
    era5_u850_file: str
    era5_u200_file: str

    # Variable names (you confirmed FLUT for NOAA)
    v_olr: str = "FLUT"
    v_u850: str = "U850"
    v_u200: str = "U200"

    # Coordinate names
    time_dim: str = "time"
    lat_dim: str = "lat"
    lon_dim: str = "lon"

    # Training period (overlap between NOAA OLR and ERA5 winds)
    train_start: str = "1986-01-01"
    train_end: str = "2017-12-31"

    # Standard WH04-ish tropical belt
    lat_range: Tuple[float, float] = (-15.0, 15.0)
    lon_range: Tuple[float, float] = (0.0, 360.0)  # assumes 0..360E convention

    # Climatology (paper-style): day-of-year mean, optionally smoothed
    doy_smooth_days: int = 31   # set <=1 to disable DOY smoothing
    drop_leap_day: bool = True  # drop Feb 29 before climatology

    # Low-frequency background removal (WH04-like)
    lowfreq_rm_days: int = 120  # subtract centered 120-day running mean of anomalies

    # EOF computation
    n_modes: int = 2
    rsvd_oversample: int = 8     # randomized SVD oversampling
    rsvd_niter: int = 2          # power iterations (0–2 usually enough)
    random_seed: int = 0

    # Memory/compute knobs
    dtype: str = "float32"       # float32 reduces memory substantially
    chunks_time: int = 180       # used for reading; doesn’t fully dask-ify compute
    chunks_lat: int = 31
    chunks_lon: int = 360

    cache_dir: str = "./rmm_cache"
    use_cache: bool = True
    overwrite_cache: bool = False

    # Optional coarse-graining (keep 1 for native 1x1; increase if needed)
    coarsen: int = 1

    # Output
    out_nc: str = "rmm_obs_eof_1986-2017_1deg.nc"


class RmmObsEofBuilder:
    """
    Build observed WH04-style EOFs and observed RMM1/RMM2 from:
      - NOAA OLR daily (FLUT)
      - ERA5 daily U850, U200

    Key preprocessing:
      - Drop Feb 29 (optional)
      - 365-day DOY climatology (contiguous 1..365) and anomalies (optional 31-day DOY smoothing)
      - Remove low-frequency background via centered running mean (default 120d)
      - Latitude weighting sqrt(cos(lat))
      - Normalize each field by one scalar std over time+space (after weighting)
    """

    def __init__(self, cfg: RmmObsEofConfig):
        self.cfg = cfg

    # -------------------------
    # Reading + subsetting
    # -------------------------
    def _open_var(self, path: str, var: str) -> xr.DataArray:
        cfg = self.cfg

        # Only chunk time at open; lat/lon rename happens after we extract the variable.
        ds = xr.open_dataset(path, chunks={cfg.time_dim: cfg.chunks_time})

        if var not in ds:
            raise KeyError(f"Variable '{var}' not found in {path}. Vars: {list(ds.data_vars)}")

        da = ds[var]

        # ---- 1) Standardize lat/lon dim names (ERA5 often uses latitude/longitude) ----
        rename_map = {}
        if "latitude" in da.dims and cfg.lat_dim not in da.dims:
            rename_map["latitude"] = cfg.lat_dim
        if "longitude" in da.dims and cfg.lon_dim not in da.dims:
            rename_map["longitude"] = cfg.lon_dim
        if rename_map:
            da = da.rename(rename_map)

        # ---- 2) Handle pressure level dim if present ----
        # Supports common ERA5 conventions: level (hPa) or plev (Pa).
        da = self._select_pressure_level_if_needed(da, var, path)

        # ---- 3) Chunk with standardized dim names (optional) ----
        chunks = {}
        if cfg.time_dim in da.dims:
            chunks[cfg.time_dim] = cfg.chunks_time
        if cfg.lat_dim in da.dims:
            chunks[cfg.lat_dim] = cfg.chunks_lat
        if cfg.lon_dim in da.dims:
            chunks[cfg.lon_dim] = cfg.chunks_lon
        if chunks:
            da = da.chunk(chunks)

        # ---- 4) Ensure standard dim order ----
        da = da.transpose(cfg.time_dim, cfg.lat_dim, cfg.lon_dim)

        return da

    def _select_pressure_level_if_needed(self, da: xr.DataArray, var: str, path: str) -> xr.DataArray:
        # Choose target based on var name or filename: 850 or 200
        def infer_target_hpa() -> Optional[int]:
            m = re.search(r"(\d{3,4})", var)
            if m:
                return int(m.group(1))
            fname = Path(path).name
            m2 = re.search(r"(\d{3,4})", fname)
            if m2:
                return int(m2.group(1))
            return None

        target_hpa = infer_target_hpa()

        # Case A: "level" in hPa
        if "level" in da.dims:
            if da.sizes["level"] == 1:
                return da.isel(level=0, drop=True)
            if target_hpa is None:
                return da.isel(level=0, drop=True)
            da = da.sel(level=target_hpa, method="nearest")
            if "level" in da.dims and da.sizes["level"] == 1:
                da = da.isel(level=0, drop=True)
            return da

        # Case B: "plev" in Pa
        if "plev" in da.dims:
            if da.sizes["plev"] == 1:
                return da.isel(plev=0, drop=True)
            if target_hpa is None:
                return da.isel(plev=0, drop=True)
            target_pa = float(target_hpa) * 100.0  # hPa -> Pa
            da = da.sel(plev=target_pa, method="nearest")
            if "plev" in da.dims and da.sizes["plev"] == 1:
                da = da.isel(plev=0, drop=True)
            return da

        return da

    def _normalize_time_to_00z(self, da: xr.DataArray) -> xr.DataArray:
        """
        Reset time stamps to a common 00Z daily coordinate (datetime64[D]),
        preserving day order and avoiding 00Z vs 09Z mismatches.
        """
        cfg = self.cfg
        t00 = np.array(da[cfg.time_dim].values, dtype="datetime64[D]").astype("datetime64[ns]")
        return da.assign_coords({cfg.time_dim: t00})

    def _ensure_lon_0_360(self, da: xr.DataArray) -> xr.DataArray:
        cfg = self.cfg
        lon = da[cfg.lon_dim]
        if lon.min() < 0:
            new_lon = (lon % 360).astype(lon.dtype)
            da = da.assign_coords({cfg.lon_dim: new_lon}).sortby(cfg.lon_dim)
        return da

    def _subset_domain(self, da: xr.DataArray) -> xr.DataArray:
        cfg = self.cfg
        da = self._ensure_lon_0_360(da)

        lat0, lat1 = cfg.lat_range
        lon0, lon1 = cfg.lon_range

        # latitude slicing (support either ascending or descending)
        lat = da[cfg.lat_dim]
        if lat[0] < lat[-1]:
            da = da.sel({cfg.lat_dim: slice(lat0, lat1)})
        else:
            da = da.sel({cfg.lat_dim: slice(lat1, lat0)})

        # longitude slicing
        lon = da[cfg.lon_dim]
        if lon1 >= 360.0:
            # strict < 360 avoids duplicated 0/360 meridian if 360 exists
            m = (lon >= lon0) & (lon < 360.0)
            da = da.sel({cfg.lon_dim: lon.where(m, drop=True)})
        else:
            da = da.sel({cfg.lon_dim: slice(lon0, lon1)})

        # Optional coarsen (mean)
        if cfg.coarsen and cfg.coarsen > 1:
            c = cfg.coarsen
            da = da.coarsen({cfg.lat_dim: c, cfg.lon_dim: c}, boundary="trim").mean()

        return da

    def _subset_time(self, da: xr.DataArray) -> xr.DataArray:
        cfg = self.cfg
        return da.sel({cfg.time_dim: slice(cfg.train_start, cfg.train_end)})

    # -------------------------
    # Preprocessing: DOY climatology + lowfreq removal
    # -------------------------
    def _drop_leap_day(self, da: xr.DataArray) -> xr.DataArray:
        cfg = self.cfg
        if not cfg.drop_leap_day:
            return da
        t = da[cfg.time_dim]
        mask = ~((t.dt.month == 2) & (t.dt.day == 29))
        return da.sel({cfg.time_dim: mask})

    def _add_doy365_coord(self, da: xr.DataArray) -> xr.DataArray:
        """
        Add contiguous 1..365 day-of-year coordinate even after dropping Feb 29.
        This avoids gaps in dayofyear (e.g., missing 60) that break DOY smoothing.
        """
        cfg = self.cfg
        t = da[cfg.time_dim]
        doy = t.dt.dayofyear
        is_leap = t.dt.is_leap_year

        # After Feb 28 in leap years (day 60+), shift down by 1 to map to 1..365.
        # Feb 29 is already removed if cfg.drop_leap_day=True.
        doy365 = xr.where(is_leap & (doy > 59), doy - 1, doy)
        return da.assign_coords(doy365=doy365)

    def _smooth_doy_climo(self, climo: xr.DataArray, win: int) -> xr.DataArray:
        """
        Smooth climatology across DOY with wraparound using a rolling mean.
        climo dims: doy365, lat, lon where doy365 is 1..365 contiguous.
        """
        if win <= 1:
            return climo

        # Expect contiguous 1..N
        doy = climo["doy365"]
        n = int(doy.max())
        pad = win // 2

        left = climo.sel(doy365=slice(n - pad + 1, n))
        right = climo.sel(doy365=slice(1, pad))

        padded = xr.concat([left, climo, right], dim="doy365")
        padded = padded.assign_coords(doy365=np.arange(1, padded.sizes["doy365"] + 1))

        sm = padded.rolling(doy365=win, center=True, min_periods=win).mean()
        sm = sm.isel(doy365=slice(pad, pad + n))
        sm = sm.assign_coords(doy365=np.arange(1, n + 1))
        return sm

    def _doy_anomaly(self, da: xr.DataArray) -> xr.DataArray:
        """
        Paper-style anomalies: remove 365-day DOY climatology (optionally smoothed).
        """
        cfg = self.cfg
        da = self._drop_leap_day(da)
        da = self._add_doy365_coord(da)

        climo = da.groupby("doy365").mean(cfg.time_dim)
        if cfg.doy_smooth_days and cfg.doy_smooth_days > 1:
            climo = self._smooth_doy_climo(climo, cfg.doy_smooth_days)

        anom = da.groupby("doy365") - climo
        # keep doy365 coordinate for diagnostics if desired
        return anom

    def _remove_lowfreq(self, anom: xr.DataArray) -> xr.DataArray:
        """
        Subtract a centered running mean from anomalies (WH04-ish lowfreq removal).
        """
        cfg = self.cfg
        w = cfg.lowfreq_rm_days
        if w is None or w <= 1:
            return anom

        low = anom.rolling({cfg.time_dim: w}, center=True, min_periods=w).mean()
        return anom - low

    def _lat_weight(self, da: xr.DataArray) -> xr.DataArray:
        cfg = self.cfg
        lat = da[cfg.lat_dim]
        w = np.sqrt(np.cos(np.deg2rad(lat)))
        return da * w

    def _normalize_by_space_time_std(self, da: xr.DataArray) -> Tuple[xr.DataArray, float]:
        """
        Normalize by a single scalar std over time+space (after weighting).
        """
        cfg = self.cfg
        std = da.std(dim=(cfg.time_dim, cfg.lat_dim, cfg.lon_dim), skipna=True)
        std_val = float(std.values)
        if not np.isfinite(std_val) or std_val == 0.0:
            raise ValueError("Computed std is invalid/zero; check data preprocessing.")
        return da / std_val, std_val

    def preprocess_one(self, da: xr.DataArray) -> Tuple[xr.DataArray, float]:
        """
        Full preprocessing for one variable:
          - DOY anomaly (365-day contiguous)
          - remove lowfreq (120d running mean)
          - lat weighting sqrt(cos(lat))
          - normalize by space-time std (scalar)
        Returns (processed, std_scalar)
        """
        da = self._doy_anomaly(da)
        da = self._remove_lowfreq(da)
        da = self._lat_weight(da)
        da, std_val = self._normalize_by_space_time_std(da)
        return da, std_val

    # -------------------------
    # Randomized SVD (economy)
    # -------------------------
    @staticmethod
    def _randomized_svd(X: np.ndarray, k: int, oversample: int = 8, niter: int = 2, seed: int = 0):
        """
        Randomized SVD for X (T,F). Returns U (T,k), S (k,), Vt (k,F).
        """
        rng = np.random.default_rng(seed)
        T, F = X.shape
        l = k + max(0, oversample)

        Omega = rng.standard_normal(size=(F, l)).astype(X.dtype, copy=False)
        Y = X @ Omega  # (T,l)

        for _ in range(max(0, niter)):
            Y = X @ (X.T @ Y)

        Q, _ = np.linalg.qr(Y, mode="reduced")  # (T,l)
        B = Q.T @ X  # (l,F)

        Ub, S, Vt = np.linalg.svd(B, full_matrices=False)
        U = Q @ Ub  # (T,l)

        U = U[:, :k]
        S = S[:k]
        Vt = Vt[:k, :]
        return U, S, Vt

    def _cache_paths(self) -> Dict[str, Path]:
        cfg = self.cfg
        d = Path(cfg.cache_dir)
        d.mkdir(parents=True, exist_ok=True)

        tag = (
            f"{cfg.train_start}_{cfg.train_end}"
            f"_lat{cfg.lat_range[0]:g}_{cfg.lat_range[1]:g}"
            f"_lon{cfg.lon_range[0]:g}_{cfg.lon_range[1]:g}"
            f"_c{cfg.coarsen}"
        )
        return {
            "olr": d / f"obs_trim_OLR_{tag}.nc",
            "u850": d / f"obs_trim_U850_{tag}.nc",
            "u200": d / f"obs_trim_U200_{tag}.nc",
        }

    def _trim_and_standardize(self, da: xr.DataArray) -> xr.DataArray:
        cfg = self.cfg

        # 1) normalize timestamps to daily 00Z
        da = self._normalize_time_to_00z(da)

        # 2) sort + drop duplicates (can appear after flooring to day)
        da = da.sortby(cfg.time_dim)
        t = da[cfg.time_dim].values
        _, idx = np.unique(t, return_index=True)
        if idx.size != t.size:
            da = da.isel({cfg.time_dim: np.sort(idx)})

        # 3) domain and time subsetting
        da = self._subset_domain(da)
        da = self._subset_time(da)

        return da.sortby(cfg.time_dim)

    def prepare_trimmed_inputs(self) -> Tuple[xr.DataArray, xr.DataArray, xr.DataArray]:
        """
        Returns trimmed (time, lat, lon) DataArrays for OLR, U850, U200.
        If cached files exist (and use_cache=True), load them.
        Otherwise generate and save them.
        """
        cfg = self.cfg
        paths = self._cache_paths()

        if (
            cfg.use_cache
            and paths["olr"].exists()
            and paths["u850"].exists()
            and paths["u200"].exists()
            and not cfg.overwrite_cache
        ):
            olr = xr.open_dataset(paths["olr"])[cfg.v_olr].load()
            u850 = xr.open_dataset(paths["u850"])[cfg.v_u850].load()
            u200 = xr.open_dataset(paths["u200"])[cfg.v_u200].load()
            return olr, u850, u200

        # Build from raw
        olr = self._open_var(cfg.noaa_olr_file, cfg.v_olr)
        u850 = self._open_var(cfg.era5_u850_file, cfg.v_u850)
        u200 = self._open_var(cfg.era5_u200_file, cfg.v_u200)

        olr = self._trim_and_standardize(olr)
        u850 = self._trim_and_standardize(u850)
        u200 = self._trim_and_standardize(u200)

        # Align to common time after standardization
        common_time = np.intersect1d(olr[cfg.time_dim].values, u850[cfg.time_dim].values)
        common_time = np.intersect1d(common_time, u200[cfg.time_dim].values)
        if common_time.size < 365 * 5:
            raise ValueError(f"Too few common days after 00Z-normalized intersection: {common_time.size}")

        olr = olr.sel({cfg.time_dim: common_time}).sortby(cfg.time_dim)
        u850 = u850.sel({cfg.time_dim: common_time}).sortby(cfg.time_dim)
        u200 = u200.sel({cfg.time_dim: common_time}).sortby(cfg.time_dim)

        # Write cache
        xr.Dataset({cfg.v_olr: olr.astype(cfg.dtype)}).to_netcdf(
            paths["olr"],
            encoding={cfg.v_olr: {"zlib": True, "complevel": 2}},
        )
        xr.Dataset({cfg.v_u850: u850.astype(cfg.dtype)}).to_netcdf(
            paths["u850"],
            encoding={cfg.v_u850: {"zlib": True, "complevel": 2}},
        )
        xr.Dataset({cfg.v_u200: u200.astype(cfg.dtype)}).to_netcdf(
            paths["u200"],
            encoding={cfg.v_u200: {"zlib": True, "complevel": 2}},
        )

        return olr, u850, u200

    # -------------------------
    # Build EOFs + RMM
    # -------------------------
    def build_and_save(self, out_nc: Optional[str] = None) -> Path:
        cfg = self.cfg
        out_path = Path(out_nc or cfg.out_nc)

        # Load trimmed + time-aligned inputs (from cache if available)
        olr, u850, u200 = self.prepare_trimmed_inputs()

        # Preprocess each
        olr_p, std_olr = self.preprocess_one(olr)
        u850_p, std_u850 = self.preprocess_one(u850)
        u200_p, std_u200 = self.preprocess_one(u200)

        # Keep only times where all fields are fully finite (after lowfreq edges etc.)
        def good_times(da: xr.DataArray) -> xr.DataArray:
            return np.isfinite(da).all(dim=(cfg.lat_dim, cfg.lon_dim))

        ok = good_times(olr_p) & good_times(u850_p) & good_times(u200_p)
        olr_p = olr_p.sel({cfg.time_dim: ok})
        u850_p = u850_p.sel({cfg.time_dim: ok})
        u200_p = u200_p.sel({cfg.time_dim: ok})

        time = olr_p[cfg.time_dim]
        if time.size < 365 * 5:
            raise ValueError(f"Too few valid days after preprocessing/edge removal: {time.size}")

        # Stack features (time, feature)
        olr_s = olr_p.stack(feature=(cfg.lat_dim, cfg.lon_dim))
        u850_s = u850_p.stack(feature=(cfg.lat_dim, cfg.lon_dim))
        u200_s = u200_p.stack(feature=(cfg.lat_dim, cfg.lon_dim))

        X = np.concatenate(
            [
                olr_s.values.astype(cfg.dtype, copy=False),
                u850_s.values.astype(cfg.dtype, copy=False),
                u200_s.values.astype(cfg.dtype, copy=False),
            ],
            axis=1,
        )

        # Safety: remove any remaining NaN rows
        row_ok = np.isfinite(X).all(axis=1)
        if not np.all(row_ok):
            X = X[row_ok, :]
            time = time.isel({cfg.time_dim: row_ok})

        U, S, Vt = self._randomized_svd(
            X,
            k=cfg.n_modes,
            oversample=cfg.rsvd_oversample,
            niter=cfg.rsvd_niter,
            seed=cfg.random_seed,
        )

        # PCs (time series): U*S
        PCs = U * S[None, :]  # (T,2)

        # Optional mode-wise std normalization (kept, because you were saving pc_std_mode)
        pc_std = PCs.std(axis=0)
        pc_std = np.where(pc_std == 0.0, np.nan, pc_std)
        PCs = PCs / pc_std[None, :]

        # WH04-style amplitude normalization: rmm_scale = sqrt(mean(RMM1^2 + RMM2^2))
        amp0 = np.sqrt(PCs[:, 0] ** 2 + PCs[:, 1] ** 2)
        scale = np.sqrt(np.nanmean(amp0 ** 2))
        if not np.isfinite(scale) or scale == 0.0:
            raise ValueError(f"Bad RMM scaling factor: {scale}")
        PCs /= scale

        # Split EOF patterns back to (mode, lat, lon) for each variable
        ngrid = olr_s.sizes["feature"]
        eof_olr = Vt[:, 0:ngrid]
        eof_u850 = Vt[:, ngrid:2 * ngrid]
        eof_u200 = Vt[:, 2 * ngrid:3 * ngrid]

        eof_olr_da = (
            xr.DataArray(
                eof_olr.astype("float32"),
                dims=("mode", "feature"),
                coords={"mode": np.arange(1, cfg.n_modes + 1), "feature": olr_s["feature"]},
                name="EOF_OLR",
            )
            .unstack("feature")
            .transpose("mode", cfg.lat_dim, cfg.lon_dim)
        )

        eof_u850_da = (
            xr.DataArray(
                eof_u850.astype("float32"),
                dims=("mode", "feature"),
                coords={"mode": np.arange(1, cfg.n_modes + 1), "feature": u850_s["feature"]},
                name="EOF_U850",
            )
            .unstack("feature")
            .transpose("mode", cfg.lat_dim, cfg.lon_dim)
        )

        eof_u200_da = (
            xr.DataArray(
                eof_u200.astype("float32"),
                dims=("mode", "feature"),
                coords={"mode": np.arange(1, cfg.n_modes + 1), "feature": u200_s["feature"]},
                name="EOF_U200",
            )
            .unstack("feature")
            .transpose("mode", cfg.lat_dim, cfg.lon_dim)
        )

        rmm1 = xr.DataArray(PCs[:, 0].astype("float32"), dims=(cfg.time_dim,), coords={cfg.time_dim: time}, name="RMM1")
        rmm2 = xr.DataArray(PCs[:, 1].astype("float32"), dims=(cfg.time_dim,), coords={cfg.time_dim: time}, name="RMM2")
        amp = xr.DataArray(np.sqrt(rmm1**2 + rmm2**2).astype("float32"), dims=(cfg.time_dim,), coords={cfg.time_dim: time}, name="RMM_amp")
        phase = xr.DataArray(np.arctan2(rmm2, rmm1).astype("float32"), dims=(cfg.time_dim,), coords={cfg.time_dim: time}, name="RMM_phase_rad")

        ds_out = xr.Dataset(
            data_vars=dict(
                EOF_OLR=eof_olr_da,
                EOF_U850=eof_u850_da,
                EOF_U200=eof_u200_da,
                RMM1=rmm1,
                RMM2=rmm2,
                RMM_amp=amp,
                RMM_phase_rad=phase,
                norm_std_olr=((), np.float32(std_olr)),
                norm_std_u850=((), np.float32(std_u850)),
                norm_std_u200=((), np.float32(std_u200)),
                rmm_scale=((), np.float32(scale)),
                sv_singular_values=(("mode",), S.astype("float32")),
                pc_std_mode=(("mode",), pc_std.astype("float32")),
            ),
            coords=dict(
                mode=np.arange(1, cfg.n_modes + 1),
                lat=olr_p[cfg.lat_dim],
                lon=olr_p[cfg.lon_dim],
                time=time,
            ),
            attrs=dict(
                description="Observed EOFs and RMM indices (WH04-like) trained from NOAA OLR + ERA5 U winds",
                train_start=cfg.train_start,
                train_end=cfg.train_end,
                lat_range=str(cfg.lat_range),
                lon_range=str(cfg.lon_range),
                grid="native (optionally coarsened)" if cfg.coarsen > 1 else "native 1deg",
                doy_smooth_days=str(cfg.doy_smooth_days),
                drop_leap_day=str(cfg.drop_leap_day),
                lowfreq_rm_days=str(cfg.lowfreq_rm_days),
                doy_definition="doy365 (1..365 contiguous; leap-year days after Feb 28 shifted by -1; Feb 29 dropped if enabled)",
                lat_weight="sqrt(cos(lat)) applied before normalization",
                normalization="each variable divided by its space-time std over training period (after lat weighting)",
                svd_method=f"randomized_svd(k={cfg.n_modes}, oversample={cfg.rsvd_oversample}, niter={cfg.rsvd_niter})",
                pc_normalization="PCs divided by their std over training period (mode-wise), then scaled by rmm_scale",
                rmm_normalization="Final scaling enforces mean(RMM1^2 + RMM2^2)=1 over training period",
                source_files=f"OLR={cfg.noaa_olr_file}; U850={cfg.era5_u850_file}; U200={cfg.era5_u200_file}",
            ),
        )

        out_path.parent.mkdir(parents=True, exist_ok=True)
        ds_out.to_netcdf(out_path)
        return out_path

In [4]:
if __name__ == "__main__":
    diag_dir = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper"
    out_dir = Path(diag_dir) / "figure_data" / "mjo"
    out_dir.mkdir(parents=True, exist_ok=True)

    # training period (overlap)
    train_start = "1986-01-01"
    train_end   = "2017-12-31"
    period = "1986-2017"

    # WH04-faithful would be 0; keep 31 only if you want and will document it
    doy_smooth_days = 30
    lowfreq_rm_days = 120
    coarsen = 1  # 1-degree resolution
    use_cache = True # save subset data first and read it if exist
    overwrite_cache = False # rewrite the subset data 

    out_nc = out_dir / f"rmm_obs_eof_1deg_{period}_climo{doy_smooth_days}_rm{lowfreq_rm_days}.nc"

    cfg = RmmObsEofConfig(
        noaa_olr_file="/lcrc/soft/climate/e3sm_diags_data/obs_for_e3sm_diags/time-series/NOAA-OLR_Daily/FLUT_198601_202112.nc",
        era5_u850_file="/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/obs_data/ERA5/daily/U850_197901_201712.nc",
        era5_u200_file="/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/obs_data/ERA5/daily/U200_197901_201712.nc",
        train_start=train_start,
        train_end=train_end,
        out_nc=str(out_nc),
        doy_smooth_days=doy_smooth_days,
        lowfreq_rm_days=lowfreq_rm_days,
        coarsen=coarsen,
        cache_dir=str(out_dir),
        use_cache=use_cache,
        overwrite_cache=overwrite_cache,
    )

    builder = RmmObsEofBuilder(cfg)
    out = builder.build_and_save()
    print(f"Wrote: {out}")


Wrote: /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_dart_paper/figure_data/mjo/rmm_obs_eof_1deg_1986-2017_climo30_rm120.nc
